In [ ]:
import numpy as np
from PGAM.gam_data_handlers import smooths_handler

# ------------------------------------------------------------
# INPUT
# ------------------------------------------------------------

spike_times = np.array([
    0.103,
    0.251,
    0.257,
    0.903,
    1.104,
    ...
])

import numpy as np

dt = 0.006  # 6 ms

# total recording duration in seconds
T_sec = recording_duration

bin_edges = np.arange(
    0,
    T_sec + dt,
    dt
)

spk, _ = np.histogram(
    spike_times,
    bins=bin_edges
)

spk = spk.astype(float)





dt = 0.006                    # 6 ms bins
spk = np.asarray(spk, dtype=float)   # spike counts, shape (T,)
T = len(spk)

# ------------------------------------------------------------
# STRICTLY CAUSAL SPIKE HISTORY
#
# hist_input[t] = spk[t-1]
#
# Therefore the "zero lag" of the PGAM temporal kernel
# corresponds to a real lag of 6 ms.
# ------------------------------------------------------------

hist_input = np.zeros_like(spk)
hist_input[1:] = spk[:-1]

# ------------------------------------------------------------
# TRIAL INDEX
# ------------------------------------------------------------

# If this is one continuous recording:
trial_idx = np.ones(T, dtype=int)

# Better, if you have actual trials:
# trial_idx[t] = trial number containing time bin t
#
# This prevents convolution from carrying spike-history effects
# across trial boundaries.

# ------------------------------------------------------------
# CREATE PGAM SMOOTH HANDLER
# ------------------------------------------------------------

sm_handler = smooths_handler()

# ------------------------------------------------------------
# SELF-HISTORY SMOOTH
#
# Desired real history:
#     6, 12, 18, 24, 30, 36 ms
#
# Because hist_input is already shifted by 6 ms,
# the PGAM temporal kernel only needs 0..30 ms relative
# to hist_input.
#
# For PGAM's centered temporal-kernel representation,
# kernel_length = 11 gives 5 bins on either side + center.
# With kernel_direction=-1 only the causal side is used.
# ------------------------------------------------------------

sm_handler.add_smooth(
    'self_history',
    [hist_input],

    ord=2,                    # temporal spline order
    knots_num=6,              # enough flexibility for a short 36-ms filter

    penalty_type='diff',
    der=2,

    is_temporal_kernel=True,
    kernel_direction=-1,      # post-spike causal effect
    kernel_length=11,

    trial_idx=trial_idx,
    time_bin=dt,

    event_input=True,
    lam=10
)


# existing place smooth
sm_handler.add_smooth(
    'self_position',
    [self_x, self_y],
    ord=4,
    knots=[x_knots, y_knots],
    penalty_type='der',
    der=2,
    is_temporal_kernel=False,
    lam=10
)

# self-history / burst smooth
hist_input = np.zeros_like(spk, dtype=float)
hist_input[1:] = spk[:-1]

sm_handler.add_smooth(
    'self_history',
    [hist_input],
    ord=2,
    knots_num=6,
    penalty_type='diff',
    der=2,
    kernel_length=11,
    kernel_direction=-1,
    trial_idx=trial_idx,
    is_temporal_kernel=True,
    time_bin=0.006,
    event_input=True,
    lam=10
)



NameError: name 'spk' is not defined

In [ ]:


gam_model = general_additive_model(
    sm_handler,
    sm_handler.smooths_var,
    spk,
    poissFam,
    fisher_scoring=False
)